In [1]:
import os, shutil
from pathlib import Path
PATH = r"D:\movies\crazywifeslut-fansly-pictures"  # Change this
MAX_SIZE = 1.5 * 1024 * 1024 * 1024  # 1 GB
EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff', '.webp')
path = Path(PATH)
files = [f for f in os.listdir(PATH) if os.path.isfile(os.path.join(PATH, f)) and f.lower().endswith(EXTS)]
idx, current_size = 1, 0

for f in files:
    f_path = os.path.join(PATH, f)
    f_size = os.path.getsize(f_path)
    
    if current_size + f_size > MAX_SIZE and current_size > 0:
        idx += 1
        current_size = 0
        
    subfolder = os.path.join(PATH, f"{path.stem}-{idx}")
    os.makedirs(subfolder, exist_ok=True)
    shutil.move(f_path, os.path.join(subfolder, f))
    current_size += f_size


In [ ]:
from selenium import webdriver

driver = webdriver.Chrome()

driver.execute_cdp_cmd(
    "Network.enable",
    {}
)

driver.execute_cdp_cmd(
    "Network.setBlockedURLs",
    {
        "urls": [
            "*.jpg",
            "*.jpeg",
            "*.png",
            "*.gif",
            "*.webp",
            "*.svg",
            "*.avif"
        ]
    }
)

driver.get("https://example.com")

In [ ]:
# model#663
import os
import json
import time
import base64
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait


urls = [

    
]


for idx, URL in enumerate(urls):
    print(f'WORKING ON URL: {idx + 1}')
    path = Path(URL)


    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(options=options)
    driver.get(URL)

    wait = WebDriverWait(driver, 15)

    count = 0
    while True:
        items = driver.find_elements(By.CSS_SELECTOR, "#albumGallery a.grid-item")
        old_count = len(items)

        buttons = driver.find_elements(By.CSS_SELECTOR, "a.js-ajax-images2")

        if not buttons:
            break

        driver.execute_script(
            "arguments[0].scrollIntoView({block:'center'});",
            buttons[0]
        )
        driver.execute_script("arguments[0].click();", buttons[0])

        try:
            wait.until(
                lambda d: len(
                    d.find_elements(By.CSS_SELECTOR, "#albumGallery a.grid-item")
                ) > old_count
            )
        except:
            break
        print(f"processed pagination: {count +1}")
        time.sleep(2)
        count+=1

    links = list(dict.fromkeys(
        x.get_attribute("href")
        for x in driver.find_elements(
            By.CSS_SELECTOR, "#albumGallery a.grid-item"
        )
        if x.get_attribute("href")
    ))

    with open(f"{path.stem}.json", "w", encoding="utf-8") as f:
        json.dump(links, f, indent=2)

    print(f"Found {len(links)} images")
driver.quit()

WORKING ON URL: 1
processed pagination: 1
processed pagination: 2
processed pagination: 3
processed pagination: 4
processed pagination: 5
Found 497 images
WORKING ON URL: 2
processed pagination: 1
processed pagination: 2
processed pagination: 3
processed pagination: 4
processed pagination: 5
Found 443 images


In [ ]:
import os
import json
import time
import random
import requests
from pathlib import Path
HOME = "https://x-fetish.tube/"

json_dir = Path(r"C:\Users\rando\Office Projects\rep_fsparse\INPUT")

for json_path in json_dir.glob("*.json"):
    

    path = Path(json_path)
    
    if not path.stem.startswith("devil"):
        continue

    OUT = fr"D:\movies\{path.stem}"
    os.makedirs(OUT, exist_ok=True)

    with open(json_path, "r", encoding="utf-8") as f:
        links = json.load(f)

    session = requests.Session()

    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/153.0.0.0 Safari/537.36"
        ),
        "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
    })

    r = session.get(HOME, timeout=30)
    r.raise_for_status()

    print("Session created")
    print("Cookies:", session.cookies.get_dict())
    print(f"Downloading {len(links)} images...\n")

    for i, url in enumerate(links, 1): 
        

        
        path = os.path.join(OUT, f"{i:05d}.jpg")

        if os.path.exists(path):
            print(f"[{i}/{len(links)}] EXISTS")
            continue

        try:
            r = session.get(
                url,
                headers={"Referer": HOME},
                timeout=30
            )

            if r.status_code == 200:
                with open(path, "wb") as f:
                    f.write(r.content)

                print(f"[{i}/{len(links)}] OK")

            else:
                print(f"[{i}/{len(links)}] HTTP {r.status_code}")

        except requests.RequestException as e:
            print(f"[{i}/{len(links)}] FAILED: {e}")

        time.sleep(random.uniform(0.3, 0.5))